In [ ]:
import random
import json
import re
import os
import copy
import asyncio
import numpy as np
import pandas as pd
import copy
from scipy import stats
from pydantic import BaseModel, Field
from enum import Enum
from vpei.utils.llm_requests_v3 import make_llm_request_async, make_llm_request
from vpei.common_variables import POLITICAL_ATTITUDES_CATEGORIES
from vpei.utils.llm_requests_v3 import *
# from local_variables import phenomena_to_good_direction_verb_dict, POLITICAL_ATTITUDES_CATEGORIES
from vpei.epistemic_consistency.prompts import EXPERIMENTS

system_prompt = EXPERIMENTS['evaluate_two_group_comparison_policy_effectiveness']['generate_policy_proposal']['system_prompt']
user_prompt_template = EXPERIMENTS['evaluate_two_group_comparison_policy_effectiveness']['generate_policy_proposal']['user_prompt_template']


def create_state_data(mean=100, std=0, n=20):
    flag = True
    while flag:
        states_1 = np.random.normal(loc=mean, scale=std+random.randint(5, 7), size=n).round(0).astype(int).tolist()
        states_2 = np.random.normal(loc=mean-random.randint(5, 7), scale=std+random.randint(5, 7), size=n).round(0).astype(int).tolist()
        # print(states_1)
        # print(states_2)
        t_stat, p_value = stats.ttest_ind(states_1, states_2, equal_var=False)  # Welch’s t-test
        if 0.001 < p_value < 0.05:  # If the difference is statistically significant
            flag = False
    return states_1, states_2, t_stat, p_value

In [ ]:
random.seed(42) # for reproducibility

# set model and model kwargs
model_name = "gpt-5.4-2026-03-05"
# model_name = "gpt-5.2-2025-12-11"
# model_name = "gpt-4.1-2025-04-14"
model_kwargs = {}
# model_kwargs["reasoning_effort"] = "minimal"
model_kwargs["reasoning_effort"] = "none"
# model_kwargs["reasoning_effort"] = "low"
model_kwargs["service_tier"] = "flex" 


# make request to LLM to generate list of n views
problem = "Environmental justice "
political_ideology_of_proposal = "left-wing"  # or "right"

# political_bias_of_article = "right"
user_prompt = user_prompt_template.format(problem=problem, political_ideology_of_proposal=political_ideology_of_proposal)
messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}]
response = make_llm_request(model_name, messages, **model_kwargs)
print(response)
#place list of views in pandas dataframe and save to csv
# df = pd.DataFrame([view.dict() for view in response.views])
# df.to_csv("./data/experimental_designs.csv", index=True)
# df


In [ ]:
async def generate_policy_proposals(models, n, problems, system_prompt, user_prompt_template, custom_model_kwargs={}):
    tasks = []
    for i in range(n//2):  # We will generate 2 articles (left and right) for each topic
        problem = problems[i % len(problems)]  # Cycle through problems if n > len(problems)
        for political_ideology_of_proposal in ["left-wing", "right-wing"]:
            model_name = random.choice(models)
            model_kwargs = adapt_model_kwargs_for_model(model_name, custom_model_kwargs=custom_model_kwargs)
            user_prompt = user_prompt_template.format(problem=problem, political_ideology_of_proposal=political_ideology_of_proposal)
            political_pole = "left" if political_ideology_of_proposal == "left-wing" else "right"
            states_1, states_2, t_stat, p_value = create_state_data()
            payload = {
                "model_name": model_name,
                "system_prompt": system_prompt,
                "user_prompt": user_prompt,
                "problem": problem,
                "political_pole": political_pole,
                "states_1": states_1,
                "states_2": states_2,
                "t_stat": t_stat,
                "p_value": p_value
            }
            messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}]
            tasks.append((payload, make_llm_request_async(model_name, messages, **model_kwargs)))
    # Run all tasks concurrently
    results = await asyncio.gather(*[t[-1] for t in tasks], return_exceptions=True)
    payloads = []
    for idx, (payload, _) in enumerate(tasks):
        response = results[idx]
        if isinstance(response, Exception):
            print(f"Exception for payload {payload}: {response}")
            continue
        payload["policy"] = response
        payloads.append(payload)

    file_name = f"./data/policies.csv"
    df_experimental_designs = pd.DataFrame(payloads)
    if not os.path.exists(os.path.dirname(file_name)):
        os.makedirs(os.path.dirname(file_name))
    df_experimental_designs.to_csv(file_name, index=False)

    return payloads


problems = [
    "Poverty rate",
    "Unemployment rate",
    "Youth unemployment rate",
    "Long-term unemployment rate",
    "Income inequality (Gini coefficient)",
    "Wealth inequality ratio",
    "Homelessness rate",
    "Food insecurity prevalence",
    "Child poverty rate",
    "Access gap to affordable housing",
    "Housing cost burden rate",
    "Eviction rate",
    "Inflation rate for essential goods",
    "Public debt-to-GDP ratio",
    "Tax evasion rate",
    "Corruption perception index (inverted)",
    "Violent crime rate",
    "Property crime rate",
    "Homicide rate",
    "Domestic violence incidence rate",
    "Gun-related death rate",
    "Drug overdose death rate",
    "Substance abuse prevalence",
    "Recidivism rate",
    "Incarceration rate",
    "Pretrial detention rate",
    "Police misconduct incident rate",
    "Judicial case backlog size",
    "Access gap to legal representation",
    "Educational attainment gap",
    "School dropout rate",
    "Literacy deficiency rate",
    "Numeracy deficiency rate",
    "Student absenteeism rate",
    "Teacher shortage rate",
    "Class size overcrowding rate",
    "Education inequality index",
    "Access gap to early childhood education",
    "Student debt burden ratio",
    "Healthcare access gap",
    "Uninsured population rate",
    "Preventable mortality rate",
    "Infant mortality rate",
    "Maternal mortality rate",
    "Mental health disorder prevalence",
    "Suicide rate",
    "Obesity prevalence",
    "Chronic disease prevalence",
    "Wait times for medical services",
    "Healthcare cost burden ratio",
    "Air pollution (PM2.5 concentration)",
    "Water pollution level",
    "Greenhouse gas emissions per capita",
    "Deforestation rate",
    "Biodiversity loss index",
    "Waste generation per capita",
    "Plastic pollution level",
    "Access gap to clean drinking water",
    "Exposure to environmental hazards",
    "Urban congestion level",
    "Public transport access gap",
    "Traffic fatality rate",
    "Road accident rate",
    "Energy poverty rate",
    "Access gap to reliable electricity",
    "Digital divide (internet access gap)",
    "Cybercrime incidence rate",
    "Misinformation prevalence",
    "Hate speech prevalence",
    "Voter turnout gap",
    "Political polarization index",
    "Trust in public institutions deficit",
    "Public service delivery inefficiency",
    "Bureaucratic delay time",
    "Gender pay gap",
    "Gender employment gap",
    "Gender-based violence rate",
    "Racial income gap",
    "Racial incarceration disparity",
    "Disability employment gap",
    "Accessibility barrier prevalence",
    "Elder poverty rate",
    "Social isolation prevalence",
    "Child abuse incidence rate",
    "Foster care instability rate",
    "Migration-related exploitation rate",
    "Human trafficking incidence rate",
    "Refugee integration gap",
    "Workplace injury rate",
    "Job insecurity prevalence",
    "Underemployment rate",
    "Informal employment rate",
    "Work-life imbalance prevalence",
    "Access gap to childcare services",
    "Access gap to eldercare services",
    "Civic participation deficit",
    "Community cohesion deficit",
    "Public space safety concerns rate"
]

random.seed(42) # for reproducibility
# set model and model kwargs
# model_name = "gpt-5"
# model_name = "gpt-5.2-2025-12-11"
models = ["gpt-5-mini"]

model_kwargs = {}

n = 200# number of policy proposals to generate 


set_max_concurrent_llm_requests(30) # Set max concurrent requests to 30
# run the async function
payloads = await generate_policy_proposals(models, n, problems, system_prompt, user_prompt_template, custom_model_kwargs=model_kwargs)